In [0]:
df_place = spark.read.format("delta").load(silver_path + "geofence_alert/")
df_tourist = spark.read.format("delta").load(silver_path + "geofence_alert/")
df_agency = spark.read.format("delta").load(silver_path + "geofence_alert/")
df_trip = spark.read.format("delta").load(silver_path + "geofence_alert/")

metrics for gold

In [0]:
from pyspark.sql.functions import *

gold_danger_zone_metrics = df_alert.join(
    df_place,
    df_alert["PlaceId"] == df_place["DangerPlaceId"],
    "left"
).groupBy(
    df_place["DangerPlaceId"],
    df_place["Name"],
    df_place["Severity"]
).agg(
    count("AlertId").alias("TotalAlerts"),
    countDistinct("TouristId").alias("UniqueTourists"),
    avg("DistanceMeters").alias("AvgDistanceMeters"),
    sum(when(col("IsResolved") == False, 1).otherwise(0)).alias("UnresolvedAlerts")
).orderBy(col("TotalAlerts").desc())